## RAG

#### Text Splitter

##### 1、CharacterTextSplitter

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
# 按段落拆分，然后合并
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=12,
    chunk_overlap=0,
)

print(splitter.split_text("第一段内容\n\n第二段内容\n\n第三段内容"))

In [ ]:
# 无分隔符时
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=5,
    chunk_overlap=0,
)

print(splitter.split_text("ABCDEFGHIJKL"))

In [ ]:
# 重叠部分
splitter = CharacterTextSplitter(
    separator="|",
    chunk_size=10,
    chunk_overlap=2,
)

print(splitter.split_text("AAAA|BBBB|CCCC|DDDD"))

In [ ]:
# 按字符分割
splitter = CharacterTextSplitter(
    separator="",
    chunk_size=5,
    chunk_overlap=2,
)

print(splitter.split_text("ABCDEFGHIJKL"))

##### 2、RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 没有换行和空格，最终按字符切分
# 默认 ["\n\n", "\n", " ", ""]
splitter = RecursiveCharacterTextSplitter(
    chunk_size=5,
    chunk_overlap=0,
)

print(splitter.split_text("ABCDEFGHIJKL"))

In [ ]:
# 短段落保留，长段落继续拆分
splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
)

print(splitter.split_text("第一段。\n\nAAAA BBBB CCCC"))

In [ ]:
# 增加额外分隔符
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "，", " ", ""],
    keep_separator="end",
    chunk_size=8,
    chunk_overlap=0,
)

print(splitter.split_text("加载文档，清理内容。生成向量，写入索引。"))

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    file_path="./assets/01-langchain-overview.txt",
    encoding="utf-8",
)

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "；", "，", " ", ""],
    keep_separator="end",
    chunk_size=100,
    chunk_overlap=10,
)

docs = loader.load()
# 分割文档对象
chunks = splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    print(f"第 {i + 1} 块，长度：{len(chunk.page_content)}")
    print(chunk.page_content)
    print("-" * 40)

##### 3、TokenTextSplitter

In [ ]:
import tiktoken
from langchain_text_splitters import TokenTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

encoding = tiktoken.get_encoding("cl100k_base")

In [ ]:
text = "Hello, world!"

token_ids = encoding.encode(text)

print("字符数：", len(text))
print("token 数：", len(token_ids))
print("token ID：", token_ids)
print("每个 token：", [encoding.decode([i]) for i in token_ids])

In [ ]:
# 按 Token 分割
splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=3,
    chunk_overlap=1,
)

print(splitter.split_text("red blue green yellow black white"))

In [ ]:
# 中文可能在 token 边界被切坏
splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=2,
    chunk_overlap=0,
)

print(splitter.split_text("你好，世界！"))


# 使用递归切分器代替
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=2,
    chunk_overlap=0,
)

print(splitter.split_text("你好，世界！"))

##### 4、chonkie SemanticChunker

In [ ]:
from chonkie import SemanticChunker, SentenceTransformerEmbeddings

In [ ]:
embeddings = SentenceTransformerEmbeddings(
    "BAAI/bge-small-zh-v1.5",
)

chunker = SemanticChunker(
    embedding_model=embeddings,
    chunk_size=256,                      # 块大小，以 token 计
    delim=["。", "！", "？", "\n"],       # 中文句子边界
    min_characters_per_sentence=1,       # 示例中允许短句独立参与分析
)

text = """
Redis 是内存键值数据库，经常被用于缓存。
缓存热点数据可以减少对数据库的访问。
合理的过期时间有助于控制缓存数据的新鲜度。
MySQL 事务可以控制一组操作的提交和回滚。
InnoDB 使用 undo 日志支持事务回滚和一致性读取。
事务隔离级别会影响并发读写时能够看到的数据。
健身之前应该进行适当的热身。
力量训练结束后需要根据训练量安排恢复。
充足的睡眠和合理的饮食有助于身体恢复。
""".strip()

chunks = chunker.chunk(text)

for i, chunk in enumerate(chunks, start=1):
    print(f"第 {i} 块，token 数：{chunk.token_count}")
    print(chunk.text)
    print("-" * 40)